In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW v_limit_positions_stddev AS
WITH latest_date AS (
    SELECT MAX(position_date) AS position_date
    FROM dbo.updated_positions
),
base AS (
    SELECT
        p.position_date,
        p.ticker,
        CAST(p.quantity AS DOUBLE) AS quantity
    FROM dbo.updated_positions p
    INNER JOIN latest_date d
        ON p.position_date = d.position_date
)
SELECT
    position_date,
    ticker,
    AVG(quantity) AS mean_qty,
    COALESCE(STDDEV_SAMP(quantity), 0.0) AS stddev_qty,
    AVG(quantity) + (2 * COALESCE(STDDEV_SAMP(quantity), 0.0)) AS position_limit_qty
FROM base
GROUP BY position_date, ticker;

In [ ]:
%%sql
CREATE OR REPLACE TABLE dbo.limit_positions_stddev AS
SELECT *
FROM v_limit_positions_stddev;

In [ ]:
%%sql
SELECT *
FROM dbo.limit_positions_stddev
ORDER BY ticker;